# The Key Equation for the Narrow Sense Reed Solomon code over $GF(16)$ 

In this notebook we develop an algorithm for BCH codes that uses the Key Equation. We develop this explicitely for the narrow sense Reed-Solomon code over $GF(16)$. 

## Python Conventions Used in This Notebook

- **Lists**: Written as `[a, b, c]`. Index from 0. `list[i]` accesses element at position $i$.
- **Functions**: Defined with `def name(args):` and called with `name(value)`.
- **Comments**: Start with `#` and are ignored by Python.
- **Operators**: `^` is XOR, `%` is modulo, `**` is exponentiation.

## Working with $GF(16)$
### Field Arithmetic



| binary | $GF(16)$ | polynomial in $\alpha$ |
|:---|:---:|---:|
|0000| 0 | 0|
|0001| 1| 1|
|0010| $\alpha$| $\alpha$|
|0100| $\alpha^{2}$| $\alpha^{2}$|
|1000| $\alpha^{3}$| $\alpha^{3}$|
|1001| $\alpha^{4}$| $\alpha^{3} + 1$|
|1011| $\alpha^{5}$| $\alpha^{3} +\alpha + 1$|
|1111| $\alpha^{6}$| $\alpha^{3}+\alpha^{2} + \alpha + 1$|
|0111| $\alpha^{7}$| $\alpha^{2} + \alpha+1$|
|1110| $\alpha^{8}$| $\alpha^{3}+\alpha^{2} + \alpha$|
|0101| $\alpha^{9}$| $\alpha^{2}+ 1$|
|1010| $\alpha^{10}$| $\alpha^{3} + \alpha$|
|1101| $\alpha^{11}$| $\alpha^{3} + \alpha^{2} + 1$|
|0011| $\alpha^{12}$| $\alpha + 1`$|
|0110| $\alpha^{13}$| $\alpha^{2} + \alpha$|
|1100| $\alpha^{14}$| $\alpha^{3} + \alpha^{2}$|


### Building Exponentiation and Logarithm Tables

To perform arithmetic efficiently in $GF(16)$, we construct two lookup tables:

- **Exponentiation table (`exp_table`)**: Maps each integer $i \in \{0, 1, \ldots, 14\}$ to the integer representation of $\alpha^i$.
- **Logarithm table (`log_table`)**: The inverse mapping—given the integer representation of a nonzero field element, it returns the exponent $i$ such that the element equals $\alpha^i$.

These tables allow us to perform multiplication and division using only addition and subtraction of exponents (modulo 15), which is much faster than polynomial multiplication and reduction.

**Python notes:**
- Lists in Python are created with square brackets: `[1, 2, 3]`
- `exp_table[i]` accesses the element at index $i$
- We extend `exp_table` to length 30 by appending a copy (minus the last element) to handle wraparound when computing $\alpha^i$ for $i \geq 15$

In [64]:
# Hardcoded mapping from power of alpha (0-14) to its binary representation
# Using your table exactly.
# exp_table[i] = integer representation of α^i

exp_table = [
    1,   # α^0  = 0001
    2,   # α^1  = 0010
    4,   # α^2  = 0100
    8,   # α^3  = 1000
    9,   # α^4  = 1001
    11,  # α^5  = 1011
    15,  # α^6  = 1111
    7,   # α^7  = 0111
    14,  # α^8  = 1110
    5,   # α^9  = 0101
    10,  # α^10 = 1010
    13,  # α^11 = 1101
    3,   # α^12 = 0011
    6,   # α^13 = 0110
    12,  # α^14 = 1100
]

# Make the exp table length 30 so we can do exp_table[i % 15] safely.
# (This is standard: duplicate for easy wraparound.)
exp_table = exp_table + exp_table[:-1]

# Now build log table (maps element -> exponent)
log_table = [None] * 16   # log_table[x] = i such that α^i = x
log_table[0] = None        # by convention log(0) is undefined

for i, val in enumerate(exp_table[:15]):   # only first cycle α^0 … α^14
    log_table[val] = i

print("exp_table:", exp_table)
print("log_table:", log_table)


exp_table: [1, 2, 4, 8, 9, 11, 15, 7, 14, 5, 10, 13, 3, 6, 12, 1, 2, 4, 8, 9, 11, 15, 7, 14, 5, 10, 13, 3, 6]
log_table: [None, 0, 1, 12, 2, 9, 13, 7, 3, 4, 10, 5, 14, 11, 8, 6]


### Testing the Tables

Let's verify our tables work correctly by checking a few values:
- $\alpha^0 = 1$ (binary 0001, integer 1)
- $\alpha^{14}$ should map to binary 1100 (integer 12)
- The logarithm of 12 should be 14
- The logarithm of 8 ($\alpha^3$) should be 3

**Python notes:**
- `print()` displays output
- Comments start with `#`

In [65]:
print(exp_table[0])   # 1
print(exp_table[1])   # 2
print(exp_table[2])   # 4
print(exp_table[3])   # 8
print(exp_table[14])  # 12 (alpha^14)

print(log_table[12])  # should be 14
print(log_table[8])   # should be 3


1
2
4
8
12
14
3


### Field Arithmetic Operations

Now we define the basic field operations using our lookup tables:

1. **Addition (`gf_add`)**: In characteristic 2, addition is bitwise XOR (exclusive or). In Python, the `^` operator performs XOR.

2. **Multiplication (`gf_mul`)**: To multiply $\alpha^i \cdot \alpha^j$, we compute $\alpha^{(i+j) \bmod 15}$.
   - Special case: if either operand is 0, return 0
   - Otherwise: look up exponents, add them mod 15, and look up the result

3. **Inverse (`gf_inv`)**: For $x = \alpha^i$, the inverse is $\alpha^{-i} = \alpha^{15-i}$ (since $\alpha^{15} = \alpha^0 = 1$).

4. **Division (`gf_div`)**: For $x = \alpha^i$ and $y = \alpha^j$, we have $x/y = \alpha^{(i-j) \bmod 15}$.

**Python notes:**
- `def function_name(parameters):` defines a function
- `if condition:` starts a conditional block
- `raise ZeroDivisionError(message)` signals an error
- `% 15` computes remainder modulo 15

In [66]:
def gf_add(a,b):
    """Addition in GF(2^4) is just XOR."""
    return a ^ b

def gf_mul(a,b):
    """Multiplication in GF(2^4) uses the exponentiation tables."""
    if a == 0 or b == 0:
        return 0
    return exp_table[(log_table[a] + log_table[b]) % 15]

def gf_inv(x):
    if x == 0:
        raise ZeroDivisionError("No inverse for 0 in GF(16)")
    # x = α^i ⇒ x⁻¹ = α^{−i} = α^{15 - i}
    return exp_table[(15 - log_table[x]) % 15]

def gf_div(x, y):
    if y == 0:
        raise ZeroDivisionError("Division by zero in GF(16)")
    if x == 0:
        return 0
    # x = α^i, y = α^j ⇒ x/y = α^{i-j}
    return exp_table[(log_table[x] - log_table[y]) % 15]

### Polynomials over $GF(16)$

We represent polynomials as **lists of coefficients in ascending degree order**. For example:
- $[1, 2, 4]$ represents $1 + 2x + 4x^2$
- $[0, 1]$ represents $x$
- $[5]$ represents the constant 5

We need helper functions to:
1. Remove leading zero coefficients (normalization)
2. Compute the degree of a polynomial

**`poly_strip(p)`**: Removes trailing zeros from the coefficient list (which correspond to leading zeros in standard polynomial notation). For example, $[1, 0, 0]$ becomes $[1]$.

**`poly_deg(p)`**: Returns the degree of the polynomial. The zero polynomial has degree $-1$ by convention.

**Python notes:**
- `len(p)` returns the length of list `p`
- `p.pop()` removes and returns the last element
- `p[-1]` accesses the last element
- `p[:]` creates a copy of list `p`

In [67]:
def poly_strip(p):
    # remove leading zeros
    while len(p) > 1 and p[-1] == 0:
        p.pop()
    return p

def poly_deg(p):
    p = poly_strip(p[:])
    return len(p) - 1 if not (len(p) == 1 and p[0] == 0) else -1

**Polynomial Addition**: Add two polynomials coefficient-wise using field addition (XOR).

**Algorithm:**
1. Create a result list of length equal to the longer polynomial
2. For each position $i$, add the $i$-th coefficients (treating missing coefficients as 0)
3. Strip trailing zeros

**Python notes:**
- `max(a, b)` returns the larger of two values
- `range(n)` generates integers from 0 to n-1
- `if i < len(p) else 0` is a conditional expression (ternary operator)

In [68]:
def poly_add(p, q):
    # add coefficient-wise over GF(16)
    n = max(len(p), len(q))
    r = [0] * n
    for i in range(n):
        a = p[i] if i < len(p) else 0
        b = q[i] if i < len(q) else 0
        r[i] = gf_add(a, b)
    return poly_strip(r)


**Scalar Multiplication**: Multiply every coefficient of a polynomial by a field element $a$.

This is used as a subroutine in polynomial multiplication and division.

In [69]:
def poly_scale(p, a):
    if a == 0:
        return [0]
    return poly_strip([gf_mul(coeff, a) for coeff in p])


**Polynomial Multiplication**: Multiply two polynomials over $GF(16)$ using the distributive law.

**Algorithm:**
For polynomials $p(x) = \sum_i p_i x^i$ and $q(x) = \sum_j q_j x^j$, the product is:
$$p(x) \cdot q(x) = \sum_{i,j} (p_i \cdot q_j) x^{i+j}$$

**Python notes:**
- `enumerate(p)` yields pairs `(index, value)` for each element
- `if a == 0: continue` skips the rest of the loop iteration
- We accumulate terms at position $i+j$ using field addition (XOR)

In [70]:
def poly_mul(p, q):
    r = [0] * (len(p) + len(q) - 1)
    for i, a in enumerate(p):
        if a == 0:
            continue
        for j, b in enumerate(q):
            if b == 0:
                continue
            r[i + j] = gf_add(r[i + j], gf_mul(a, b))
    return poly_strip(r)


**Polynomial Division**: Divide polynomial `num` (numerator) by `den` (denominator) to get quotient and remainder.

This implements the standard polynomial long division algorithm, adapted for $GF(16)$:

1. If the degree of the remainder is less than the degree of the divisor, we're done
2. Otherwise:
   - Divide the leading coefficient of the remainder by the leading coefficient of the divisor
   - Multiply the divisor by this ratio and the appropriate power of $x$
   - Subtract from the remainder (which is the same as addition in characteristic 2)
   - Repeat

**Returns:** A tuple `(quotient, remainder)` such that `num = quotient * den + remainder` and $\deg(\text{remainder}) < \deg(\text{divisor})$.

**Python notes:**
- `num[:]` creates a copy to avoid modifying the original
- The loop continues while the remainder's degree is at least the divisor's degree

In [71]:
def poly_divmod(num, den):
    num = poly_strip(num[:])
    den = poly_strip(den[:])
    if den == [0]:
        raise ZeroDivisionError("Polynomial division by zero")

    deg_den = poly_deg(den)
    lead_den = den[-1]

    inv_lead_den = gf_inv(lead_den)
    quotient = [0] * max(1, len(num) - len(den) + 1)
    remainder = num[:]

    while poly_deg(remainder) >= deg_den and remainder != [0]:
        deg_rem = poly_deg(remainder)
        lead_rem = remainder[-1]

        # monomial factor
        factor_deg = deg_rem - deg_den
        factor_coeff = gf_mul(lead_rem, inv_lead_den)

        # update quotient
        quotient[factor_deg] = gf_add(quotient[factor_deg], factor_coeff)

        # subtract factor * den from remainder
        sub_poly = [0] * (factor_deg + len(den))
        for i, c in enumerate(den):
            sub_poly[factor_deg + i] = gf_mul(factor_coeff, c)

        remainder = poly_add(remainder, sub_poly)  # subtraction = addition

    return poly_strip(quotient), poly_strip(remainder)


## The Narrow-Sense Reed-Solomon Code over $GF(16)$

### Code Parameters

We now construct a **narrow-sense Reed-Solomon code** over $GF(16)$ with designed distance $\delta = 5$.

**Definition:** A narrow-sense RS code has:
- **Length** $n = q - 1 = 15$ (one less than the field size)
- **Designed distance** $\delta$ (the minimum number of consecutive powers of $\alpha$ that are roots of all codeword polynomials)
- **Dimension** $k = n - \delta + 1 = 15 - 5 + 1 = 11$

This code can correct up to $t = \lfloor \frac{\delta - 1}{2} \rfloor = 2$ errors.

**Codeword structure:** A message polynomial $m(x)$ of degree at most $k-1 = 10$ is encoded as:
$$c(x) = m(x) \cdot g(x)$$
where $g(x)$ is the **generator polynomial**.

### Generator Polynomial

The generator polynomial is the lowest-degree monic polynomial whose roots are $\alpha, \alpha^2, \ldots, \alpha^{\delta-1}$:
$$g(x) = (x - \alpha)(x - \alpha^2)(x - \alpha^3)(x - \alpha^4)$$

**Python notes:**
- We build $g(x)$ by multiplying the factors $(x - \alpha^i)$ one at a time
- Each factor is represented as a list: $[c_0, c_1]$ means $c_0 + c_1 x$
- So $(x - \alpha^i)$ is $[-\alpha^i, 1]$ but since we're in characteristic 2, $-\alpha^i = \alpha^i$

In [72]:
# Build generator polynomial g(x) = (x - α)(x - α^2)(x - α^3)(x - α^4)
# Start with g(x) = 1
g = [1]

# Multiply by each factor (x - α^i) for i = 1, 2, 3, 4
for i in range(1, 5):
    # In characteristic 2, -α^i = α^i
    alpha_i = exp_table[i]
    factor = [alpha_i, 1]  # represents (x - α^i) = α^i + x
    g = poly_mul(g, factor)

print("Generator polynomial g(x):")
print("Coefficients (ascending degree):", g)
print("Degree:", poly_deg(g))

Generator polynomial g(x):
Coefficients (ascending degree): [10, 3, 9, 7, 1]
Degree: 4


### Understanding the Generator Polynomial

The generator polynomial $g(x)$ has degree $\delta - 1 = 4$, so it has 5 coefficients.

**Reading the coefficients:** The output `[10, 3, 9, 7, 1]` gives the integer representations of the field elements (refer to the table at the beginning). We can write this as a polynomial over $GF(16)$:

$$g(x) = 10 + 3x + 9x^2 + 7x^3 + x^4$$

Or, using the polynomial representations from our table:
- $10 = \alpha^{10}$ corresponds to $\alpha^3 + \alpha$
- $3 = \alpha^{12}$ corresponds to $\alpha + 1$
- $9 = \alpha^4$ corresponds to $\alpha^3 + 1$
- $7 = \alpha^7$ corresponds to $\alpha^2 + \alpha + 1$
- $1 = 1$ corresponds to $1$

So we can also write:
$$g(x) = (\alpha^3 + \alpha) + (\alpha + 1)x + (\alpha^3 + 1)x^2 + (\alpha^2 + \alpha + 1)x^3 + x^4$$

## The Key Equation Algorithm

### Algorithm Steps
**For our code:** $\delta = 5$, $t = 2$, $n = 15$.
#### Step 1: Compute Syndromes

Upon receiving the word $y$, evaluate the syndromes $S_1, S_2, \ldots, S_{\delta-1}$ where $S_j = y(\alpha^j)$ for all $j$.

If $S_1 = S_2 = \cdots = S_{\delta-1} = 0$, then $y$ is a codeword and we decode as $y$; otherwise proceed to Step 2.

In [73]:
def syndrome_calc(received, delta=5):
    """Compute syndromes S_1, S_2, ..., S_{delta-1} for received polynomial."""
    syndromes = []
    for j in range(1, delta):
        # Evaluate received at α^j
        alpha_j = exp_table[j]
        S_j = 0
        for i, coeff in enumerate(received):
            term = gf_mul(coeff, exp_table[(log_table[alpha_j] * i) % 15]) if coeff != 0 else 0
            S_j = gf_add(S_j, term)
        syndromes.append(S_j)
    return syndromes

In [74]:
x = [0]*15
Sx = syndrome_calc(x, delta=5)
print("Syndromes:", Sx)

y= [10,3,9,7,1,0,0,0,0,0,0,0,0,0,0]

Sy = syndrome_calc(y, delta=5)   
print("Syndromes:", Sy)

z= [10,3,9,7,1,0,1,0,1,0,0,0,0,0,0]

Sz = syndrome_calc(z, delta=5)
print("Syndromes:", Sz)

Syndromes: [0, 0, 0, 0]
Syndromes: [0, 0, 0, 0]
Syndromes: [1, 1, 13, 1]


#### Step 2: Form Syndrome Polynomial and Run Extended Euclidean Algorithm

Put $S(x) = 1 + S_1 x + S_2 x^2 + \cdots + S_{2t} x^{2t}$.

Run the Euclidean Algorithm with $a(x) = x^{2t+1}$ and $b(x) = S(x)$ until at step $j$ we have $\deg(r_j(x)) \leq t$.

Then put:
$$s(x) = \frac{1}{v_j(0)} v_j(x) \quad \text{and} \quad \omega(x) = \frac{1}{v_j(0)} r_j(x)$$


In [75]:
def euclidean_algorithm(S, delta=5, t=2):
    """Run Extended Euclidean Algorithm to find error locator and evaluator polynomials."""
    # Initialize: a(x) = x^{2t+1}, b(x) = S(x)
    a = [0] * (2 * t + 2)
    a[-1] = 1  # x^{2t+1}
    b = [1] + S
    if len(b) >2 * t:
        b = b[:2*t+1]

    u_minus_2 = [1]  # u_{-2}(x) = 1
    u_minus_1 = [0]  # u_{-1}(x) = 0
    v_minus_2 = [0]  # v_{-2}(x) = 0
    v_minus_1 = [1]  # v_{-1}(x) = 1
    r_minus_2 = a
    r_minus_1 = b

    q,r = poly_divmod(r_minus_2, r_minus_1)

    u = poly_add(u_minus_2, poly_mul(q, u_minus_1))
    v = poly_add(v_minus_2, poly_mul(q, v_minus_1))
    r = poly_add(r_minus_2, poly_mul(q, r_minus_1))
    while poly_deg(r) > t:
        r_minus_2, r_minus_1 = r_minus_1, r
        u_minus_2, u_minus_1 = u_minus_1, u
        v_minus_2, v_minus_1 = v_minus_1, v

        q,r = poly_divmod(r_minus_2, r_minus_1)

        u = poly_add(u_minus_2, poly_mul(q, u_minus_1))
        v = poly_add(v_minus_2, poly_mul(q, v_minus_1))
    
    error_locator = poly_scale(v, gf_inv(v[0]))
    error_evaluator = poly_scale(r, gf_inv(v[0]))
    return error_locator, error_evaluator



In [76]:
error_locator, error_evaluator = euclidean_algorithm(Sz, delta=5, t=2)
print("Error Locator Polynomial:", error_locator)
print("Error Evaluator Polynomial:", error_evaluator)

Error Locator Polynomial: [1, 1, 12]
Error Evaluator Polynomial: [1, 0, 12]


#### Step 3: Find Error Locations

Find $0 \leq i_1 < i_2 < \cdots < i_k \leq n-1$ such that $X_1 = \alpha^{i_1}, X_2 = \alpha^{i_2}, \ldots, X_k = \alpha^{i_k}$ are the reciprocals of the zeros of $s(x)$.


In [77]:
def poly_eval(p, x):
    """Evaluate polynomial p at field element x."""
    result = 0
    for i, coeff in enumerate(p):
        if coeff == 0:
            continue
        # Compute coeff * x^i
        if i == 0:
            term = coeff
        else:
            term = gf_mul(coeff, exp_table[(log_table[x] * i) % 15])
        result = gf_add(result, term)
    return result

def find_roots(p, n=15):
    """Find all roots of polynomial p in GF(16).
    
    Returns list of field elements that are roots.
    """
    roots = []
    for i in range(1, n+1):  # Check α^0, α^1, ..., α^{n-1}
        element = exp_table[i % 15]
        if poly_eval(p, element) == 0:
            roots.append(element)
    return roots

In [78]:
roots = find_roots(error_locator, n=15)
print("Roots of Error Locator Polynomial:", roots)

Roots of Error Locator Polynomial: [7, 5]


In [79]:
X_i = [gf_inv(root) for root in roots]
print("Inverses of Roots (X_i):", X_i)
print("Location of Errors (in terms of powers of α):", [log_table[x] for x in X_i])

Inverses of Roots (X_i): [14, 15]
Location of Errors (in terms of powers of α): [8, 6]



#### Step 4: Calculate Error Sizes

Calculate the error values:
$$Y_i = \frac{X_i \omega(X_i^{-1})}{\prod_{j=1, j \neq i}^k (X_i - X_j)} \quad \text{for } i = 1, 2, \ldots, k$$



In [80]:
def calculate_error_magnitudes(X_i, error_evaluator):
    """Calculate error magnitudes."""
    error_magnitudes = []
    for X in X_i:
        denominator = 1
        # Compute Ω(X^{-1})
        Omega_at_X_inv = poly_eval(error_evaluator, gf_inv(X))
        numerator = gf_mul(X, Omega_at_X_inv)
        for X_j in X_i:
            if X_j == X:
                continue
            term = gf_add(X, X_j)
            denominator = gf_mul(denominator, term)
        error_magnitude = gf_div(numerator, denominator)
        error_magnitudes.append((log_table[X],error_magnitude))
    return error_magnitudes


#### Step 5: Decode

Put $e = (0, \ldots, 0, \underbrace{Y_1}_{\text{position } i_1}, 0, \ldots, 0, \underbrace{Y_2}_{\text{position } i_2}, 0, \ldots, 0, \underbrace{Y_k}_{\text{position } i_k}, 0, \ldots, 0)$ and decode as $c := y - e$.



In [81]:
e = [0]*15
error_info = calculate_error_magnitudes(X_i, error_evaluator)
for loc, mag in error_info:
    e[loc] = mag
print("Error vector e:", e)

Error vector e: [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0]


In [82]:
codeword = poly_add(z, e)
print("Corrected codeword:", codeword)

Corrected codeword: [10, 3, 9, 7, 1]


In [83]:
poly_divmod(codeword, g)

([1], [0])

In [84]:
def key_equation_alg(y, delta=5, t=2):
    syndromes = syndrome_calc(y, delta)
    if all(s == 0 for s in syndromes):
        return y  # No errors detected
    error_locator, error_evaluator = euclidean_algorithm(syndromes, delta, t)
    error_vector = [0]*15
    X_i = [gf_inv(root) for root in find_roots(error_locator)]
    error_info = calculate_error_magnitudes(X_i, error_evaluator)
    for loc, mag in error_info:
        error_vector[loc] = mag
    return poly_add(y, error_vector)

In [85]:
key_equation_alg(z)

[10, 3, 9, 7, 1]